In [14]:
# GOBERNANZA Y ÉTICA DE DATOS - TRÁFICO DE RED
import pandas as pd
import numpy as np
import hashlib
from pathlib import Path

print("Librerias importadas ------")

# CREAR CARPETAS PARA GOBERNANZA
BASE_DIR = Path("/content/")
DATA_GOV = BASE_DIR / "datos" / "gobernanza"
DOCS_DIR = BASE_DIR / "documentos"

DATA_GOV.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Datos gobernanza: {DATA_GOV}")
print(f"Documentos: {DOCS_DIR}")

Librerias importadas ------
Datos gobernanza: /content/datos/gobernanza
Documentos: /content/documentos


In [15]:
# CARGAR DATASET REAL (PARQUET)
ruta_parquet = BASE_DIR / "dataset_crudo_limpio_final.parquet"

try:
    df_red = pd.read_parquet(ruta_parquet)
    print(f"Dataset de red cargado. Forma: {df_red.shape}")
    print(df_red.head(3))
except Exception as e:
    print(f"Error al cargar archivo: {e}")

Dataset de red cargado. Forma: (2830540, 84)
                               flow_id     source_ip  source_port  \
0  192.168.10.3-192.168.10.9-88-1031-6  192.168.10.9         1031   
1  192.168.10.9-69.31.33.224-1057-80-6  192.168.10.9         1057   
2  192.168.10.3-192.168.10.9-88-1058-6  192.168.10.9         1058   

  destination_ip  destination_port  protocol           timestamp  \
0   192.168.10.3                88         6 2017-07-03 08:56:38   
1   69.31.33.224                80         6 2017-07-03 08:57:01   
2   192.168.10.3                88         6 2017-07-03 08:57:04   

   flow_duration  total_fwd_packets  total_backward_packets  ...  \
0          609.0                  7                       4  ...   
1        40643.0                  3                       4  ...   
2         1067.0                  9                       6  ...   

   min_seg_size_forward  active_mean  active_std  active_max  active_min  \
0                  20.0          0.0         0.0        

In [16]:
# CLASIFICACIÓN DE DATOS DE RED

clasificacion = pd.DataFrame([
    ["flow_id", "Identificador interno", "Confidencial", "Vincular registros", "Seudonimizar"],
    ["source_ip", "Identificador directo", "Confidencial", "Identificación de origen", "Enmascarar octetos"],
    ["destination_ip", "Identificador directo", "Confidencial", "Identificación de destino", "Enmascarar octetos"],
    ["source_port", "Infraestructura", "Interno", "Identificar servicio emisor", "Enmascarar puerto"],
    ["destination_port", "Infraestructura", "Público", "Categorización de servicio", "Conservar"],
    ["protocol", "Técnico/Red", "Público", "Caracterización de tráfico", "Conservar"],
    ["flow_duration", "Métrica de red", "Interno", "Detección de anomalías", "Generalizar por rango"],
    ["total_fwd_packets", "Métrica de red", "Interno", "Medición de volumen", "Conservar"],
    ["min_seg_size_forward", "Métrica de red", "Interno", "Análisis de encabezados", "Conservar"],
    ["active_mean", "Métrica de red", "Interno", "Medición de actividad", "Conservar"],
    ["idle_mean", "Métrica de red", "Interno", "Análisis de pausas", "Generalizar por rango"],
    ["label", "Categoría objetivo", "Público", "Clasificación de ataques", "Conservar"]
], columns=["campo", "tipo", "clasificacion", "finalidad", "control"])

print("Matriz de clasificación ----------")
print(clasificacion)

# GUARDAR CLASIFICACIÓN
ruta_clasificacion = DOCS_DIR / "clasificacion_datos_redes.xlsx"
clasificacion.to_excel(ruta_clasificacion, index=False)

print(f"Clasificación guardada: {ruta_clasificacion}")

Matriz de clasificación ----------
                   campo                   tipo clasificacion  \
0                flow_id  Identificador interno  Confidencial   
1              source_ip  Identificador directo  Confidencial   
2         destination_ip  Identificador directo  Confidencial   
3            source_port        Infraestructura       Interno   
4       destination_port        Infraestructura       Público   
5               protocol            Técnico/Red       Público   
6          flow_duration         Métrica de red       Interno   
7      total_fwd_packets         Métrica de red       Interno   
8   min_seg_size_forward         Métrica de red       Interno   
9            active_mean         Métrica de red       Interno   
10             idle_mean         Métrica de red       Interno   
11                 label     Categoría objetivo       Público   

                      finalidad                control  
0            Vincular registros           Seudonimizar  
1    

In [17]:
# SEUDONIMIZACIÓN

SALT = "proyecto-gobernanza-redes"

def tokenizar(valor):
    """Convierte un valor en un código único de 12 caracteres."""
    texto = f"{SALT}|{valor}".encode('utf-8')
    return hashlib.sha256(texto).hexdigest()[:12]

if 'flow_id' in df_red.columns:
    df_red["flow_token"] = df_red["flow_id"].astype(str).apply(tokenizar)
    print("========Seudonimización aplicada=========")
    print(df_red[["flow_id", "flow_token"]].head())

========Seudonimización aplicada=========
                                 flow_id    flow_token
0    192.168.10.3-192.168.10.9-88-1031-6  a08c3a8294d5
1    192.168.10.9-69.31.33.224-1057-80-6  733cd6ff3fd9
2    192.168.10.3-192.168.10.9-88-1058-6  33b7cc67f306
3   192.168.10.3-192.168.10.9-389-1060-6  b2a4709a6ab6
4  192.168.10.3-192.168.10.17-88-35499-6  778a46e0c187


In [18]:
# ENMASCARAMIENTO DE IP Y PUERTO

def enmascarar_ip(ip):
    """Oculta los últimos tres octetos de la IP"""
    partes = str(ip).split('.')
    if len(partes) == 4:
        return f"{partes[0]}.***.***.***"
    return "***.***.***.***"

if 'source_ip' in df_red.columns:
    df_red["source_ip_mask"] = df_red["source_ip"].apply(enmascarar_ip)

if 'destination_ip' in df_red.columns:
    df_red["destination_ip_mask"] = df_red["destination_ip"].apply(enmascarar_ip)

if 'source_port' in df_red.columns:
    df_red["source_port_mask"] = df_red["source_port"].astype(str).apply(
        lambda x: "***" + x[-2:] if len(x) >= 2 else "***"
    )

print("IPs y Puertos enmascarados-----------")
print(df_red[["source_ip", "source_ip_mask", "destination_ip", "destination_ip_mask"]].head())

IPs y Puertos enmascarados-----------
       source_ip   source_ip_mask destination_ip destination_ip_mask
0   192.168.10.9  192.***.***.***   192.168.10.3     192.***.***.***
1   192.168.10.9  192.***.***.***   69.31.33.224      69.***.***.***
2   192.168.10.9  192.***.***.***   192.168.10.3     192.***.***.***
3   192.168.10.9  192.***.***.***   192.168.10.3     192.***.***.***
4  192.168.10.17  192.***.***.***   192.168.10.3     192.***.***.***


In [19]:
# GENERALIZACIÓN DE DURACIÓN E IDLE

bins_dur = [-1, 100, 1000, 10000, 100000, 1e9]
labels_dur = ["<=100", "101-1k", "1k-10k", "10k-100k", ">100k"]

if 'flow_duration' in df_red.columns:
    df_red["rango_duracion"] = pd.cut(df_red["flow_duration"], bins=bins_dur, labels=labels_dur)

bins_idle = [-1, 0, 1000, 100000, 1e9]
labels_idle = ["Sin Reposo", "Bajo", "Medio", "Alto"]

if 'idle_mean' in df_red.columns:
    df_red["rango_reposo"] = pd.cut(df_red["idle_mean"], bins=bins_idle, labels=labels_idle)

print("Duración generalizada: ---------------")
print(df_red[["flow_duration", "rango_duracion"]].head())

Duración generalizada: ---------------
   flow_duration rango_duracion
0          609.0         101-1k
1        40643.0       10k-100k
2         1067.0         1k-10k
3         3536.0         1k-10k
4         1135.0         1k-10k


In [20]:
# CREAR VISTA ANALÍTICA PROTEGIDA
# PRINCIPIO DE MINIMIZACIÓN

columnas_deseadas = [
    "flow_token",
    "source_ip_mask",
    "destination_ip_mask",
    "source_port_mask",
    "destination_port",
    "protocol",
    "rango_duracion",
    "rango_reposo",
    "total_fwd_packets",
    "min_seg_size_forward",
    "active_mean",
    "label"
]

cols_existentes = [col for col in columnas_deseadas if col in df_red.columns]
vista_analitica = df_red[cols_existentes].copy()

print("VISTA ANALÍTICA PROTEGIDA (SIN IDENTIFICADORES DIRECTOS)")
print(vista_analitica.head())

VISTA ANALÍTICA PROTEGIDA (SIN IDENTIFICADORES DIRECTOS)
     flow_token   source_ip_mask destination_ip_mask source_port_mask  \
0  a08c3a8294d5  192.***.***.***     192.***.***.***            ***31   
1  733cd6ff3fd9  192.***.***.***      69.***.***.***            ***57   
2  33b7cc67f306  192.***.***.***     192.***.***.***            ***58   
3  b2a4709a6ab6  192.***.***.***     192.***.***.***            ***60   
4  778a46e0c187  192.***.***.***     192.***.***.***            ***99   

   destination_port  protocol rango_duracion rango_reposo  total_fwd_packets  \
0                88         6         101-1k   Sin Reposo                  7   
1                80         6       10k-100k   Sin Reposo                  3   
2                88         6         1k-10k   Sin Reposo                  9   
3               389         6         1k-10k   Sin Reposo                 13   
4                88         6         1k-10k   Sin Reposo                  9   

   min_seg_size_forward

In [21]:
# GUARDAR VISTA PROTEGIDA

ruta_publicable = DATA_GOV / "dataset_publicable_redes.csv"
ruta_parquet_pub = DATA_GOV / "dataset_publicable_redes.parquet"

vista_analitica.to_csv(ruta_publicable, index=False, encoding="utf-8")
vista_analitica.to_parquet(ruta_parquet_pub, index=False)

print(f"Vista protegida guardada: {ruta_publicable}")

Vista protegida guardada: /content/datos/gobernanza/dataset_publicable_redes.csv


In [22]:
# VALIDACIÓN AUTOMÁTICA

identificadores_directos = {"flow_id", "source_ip", "destination_ip"}
expuestos = identificadores_directos.intersection(vista_analitica.columns)

print("Verificando identificadores directos: --------------")
print(f"Identificadores directos expuestos: {expuestos}")

if len(expuestos) == 0:
    print("Ningún identificador directo expuesto!!!!")
else:
    print("ALERTA: ¡Identificadores directos expuestos!")

Verificando identificadores directos: --------------
Identificadores directos expuestos: set()
Ningún identificador directo expuesto!!!!


In [23]:
# MATRIZ DE RIESGOS

riesgos = pd.DataFrame([
    ["Exposición de direcciones IP internas", 4, 5, "Enmascarar octetos de IP", "Líder de datos", "Dataset protegido"],
    ["Reidentificación por fingerprinting", 3, 4, "Generalizar duración e idle", "Analista", "Análisis de grupos"],
    ["Acceso excesivo al dataset crudo", 3, 5, "RBAC y mínimo privilegio", "Administrador", "Matriz de acceso"],
    ["Retención indefinida de logs", 3, 3, "Regla de retención", "Líder de proyecto", "Bitácora de eliminación"],
    ["Uso del dato para finalidad no definida", 2, 5, "Registrar propósito", "Propietario del dato", "Ficha de finalidad"],
    ["Conclusiones sesgadas por tráfico", 3, 4, "Revisión ética y balanceo", "Equipo analítico", "Reflexión y pruebas"]
], columns=["riesgo", "probabilidad", "impacto", "control", "responsable", "evidencia"])

riesgos["puntaje"] = riesgos["probabilidad"] * riesgos["impacto"]
riesgos["nivel"] = riesgos["puntaje"].apply(
    lambda x: "Crítico" if x >= 16 else "Alto" if x >= 10 else "Moderado" if x >= 5 else "Bajo"
)

print("MATRIZ DE RIESGOS:")
print(riesgos.sort_values("puntaje", ascending=False))

# GUARDAR LA MATRIZ DE RIESGOS
ruta_riesgos = DOCS_DIR / "matriz_riesgos_redes.xlsx"
riesgos.to_excel(ruta_riesgos, index=False)

print(f"Matriz de riesgos guardada---------- {ruta_riesgos}")

MATRIZ DE RIESGOS:
                                    riesgo  probabilidad  impacto  \
0    Exposición de direcciones IP internas             4        5   
2         Acceso excesivo al dataset crudo             3        5   
1      Reidentificación por fingerprinting             3        4   
5        Conclusiones sesgadas por tráfico             3        4   
4  Uso del dato para finalidad no definida             2        5   
3             Retención indefinida de logs             3        3   

                       control           responsable                evidencia  \
0     Enmascarar octetos de IP        Líder de datos        Dataset protegido   
2     RBAC y mínimo privilegio         Administrador         Matriz de acceso   
1  Generalizar duración e idle              Analista       Análisis de grupos   
5    Revisión ética y balanceo      Equipo analítico      Reflexión y pruebas   
4          Registrar propósito  Propietario del dato       Ficha de finalidad   
3          

In [24]:
# POLÍTICA BREVE DE GOBERNANZA

politica = """
# Política de Gobernanza - Network Security Analytics

## 1. Propósito
Definir cómo se utilizan, protegen y comparten los datos de tráfico de red del proyecto.

## 2. Clasificación
- Público: puertos estándar, etiquetas de tráfico (label)
- Interno: métricas de volumen y tiempos (duration, active, idle)
- Confidencial: direcciones IP (origen y destino), flow_id
- Restringido: claves de salado (SALT) y logs crudos

## 3. Acceso por Rol
- Administrador: acceso total a datos crudos
- Analista / Científico de datos: solo vista protegida (sin identificadores)
- Auditor: solo lectura de evidencias

## 4. Minimización
Solo se incluyen los campos estrictamente necesarios para el entrenamiento de modelos IDS.

## 5. Retención
Los datos se conservan durante el ciclo del proyecto y se eliminan al finalizar.

## 6. Ética
No se utilizan direcciones IP para desanonimizar equipos ni usuarios en la red.
"""

# GUARDAR POLÍTICA
ruta_politica = DOCS_DIR / "politica_gobernanza_redes.md"
with open(ruta_politica, "w", encoding="utf-8") as f:
    f.write(politica)

print(f"Política guardada: {ruta_politica}")

Política guardada: /content/documentos/politica_gobernanza_redes.md


In [25]:
# VERIFICAR ARCHIVOS GENERADOS
print("====ARCHIVOS GENERADOS====")
print("=" * 40)
!ls -la /content/datos/gobernanza/
print("====DOCUMENTOS====")
!ls -la /content/documentos/

====ARCHIVOS GENERADOS====
total 308856
drwxr-xr-x 2 root root      4096 Sep 23 03:51 .
drwxr-xr-x 3 root root      4096 Sep 23 03:42 ..
-rw-r--r-- 1 root root 263139449 Sep 23 05:12 dataset_publicable_redes.csv
-rw-r--r-- 1 root root  53115621 Sep 23 05:12 dataset_publicable_redes.parquet
====DOCUMENTOS====
total 40
drwxr-xr-x 3 root root 4096 Sep 23 05:11 .
drwxr-xr-x 1 root root 4096 Sep 23 03:43 ..
-rw-r--r-- 1 root root 5604 Sep 23 03:50 clasificacion_datos_redes_Andres_Gonzalez.xlsx
-rw-r--r-- 1 root root 5605 Sep 23 05:11 clasificacion_datos_redes.xlsx
drwxr-xr-x 2 root root 4096 Sep 23 03:53 .ipynb_checkpoints
-rw-r--r-- 1 root root 5600 Sep 23 05:12 matriz_riesgos_redes.xlsx
-rw-r--r-- 1 root root  915 Sep 23 05:12 politica_gobernanza_redes.md
